# Compare cost only after candidates pass quality gates

This credential-free lab adapts MLflow's [cost-quality trade-off cookbook](https://mlflow.org/cookbook/cost-quality-tradeoff/) to logical platform resources and governed evidence.

The cookbook's durable idea is to hold the prompt, cases, scorers, and inference settings fixed while changing only the model. This repository does not embed vendor model IDs or a price table in application code. Use trace-recorded cost when available and the platform gateway or billing system as the chargeback source of truth.

## 1. Freeze the comparison contract

A model comparison is interpretable only when every other material input is identical. Evaluation-judge cost is reported separately from target-inference cost so a more expensive rubric cannot masquerade as a more expensive application model.

In [ ]:
import hashlib
import json

PROMPT_URI = "fixture:earnings_summary/v2"
PROMPT_TEMPLATE_FIXTURE = (
    "Summarize only the supplied fictional facts. "
    "Include {{source_id}} exactly once and provide no investment advice."
)
PROMPT_DIGEST = hashlib.sha256(PROMPT_TEMPLATE_FIXTURE.encode("utf-8")).hexdigest()
CASE_IDS = [
    "quarterly-revenue-and-margin",
    "forward-revenue-and-margin-guidance",
    "cash-flow-inventory-and-supplier-risk",
]
DATASET_DIGEST = hashlib.sha256(
    json.dumps(CASE_IDS, separators=(",", ":")).encode("utf-8")
).hexdigest()
INFERENCE_PARAMETERS = {"temperature": 0.0, "max_tokens": 400}
SCORER_SET = "earnings-release-scorers-v1"

{
    "prompt_uri": PROMPT_URI,
    "prompt_digest": PROMPT_DIGEST,
    "dataset_digest": DATASET_DIGEST,
    "inference_parameters": INFERENCE_PARAMETERS,
    "scorer_set": SCORER_SET,
}

## 2. Inspect quality, policy, latency, tokens, cost, and coverage together

The following values are explicitly labelled `simulated_offline_fixture`; they demonstrate decision shape, not provider performance or current pricing. `quality-chat` intentionally has incomplete cost evidence.

In [ ]:
import pandas as pd

MEASUREMENTS = [
    {
        "logical_model": "economy-chat",
        "quality_score": 0.92,
        "minimum_row_quality": 0.90,
        "critical_case_pass_rate": 1.0,
        "recommendation_policy_compliance": 1.0,
        "latency_ms_mean": 340.0,
        "input_tokens": 300,
        "output_tokens": 180,
        "target_inference_cost_usd": 0.0034,
        "evaluation_judge_cost_usd": 0.0018,
        "cost_coverage": 1.0,
        "measurement_source": "simulated_offline_fixture",
    },
    {
        "logical_model": "general-chat",
        "quality_score": 0.97,
        "minimum_row_quality": 0.95,
        "critical_case_pass_rate": 1.0,
        "recommendation_policy_compliance": 1.0,
        "latency_ms_mean": 520.0,
        "input_tokens": 302,
        "output_tokens": 210,
        "target_inference_cost_usd": 0.0068,
        "evaluation_judge_cost_usd": 0.0018,
        "cost_coverage": 1.0,
        "measurement_source": "simulated_offline_fixture",
    },
    {
        "logical_model": "quality-chat",
        "quality_score": 0.99,
        "minimum_row_quality": 0.97,
        "critical_case_pass_rate": 1.0,
        "recommendation_policy_compliance": 1.0,
        "latency_ms_mean": 710.0,
        "input_tokens": 301,
        "output_tokens": 235,
        "target_inference_cost_usd": None,
        "evaluation_judge_cost_usd": 0.0018,
        "cost_coverage": 0.67,
        "measurement_source": "simulated_offline_fixture",
    },
]

comparison = pd.DataFrame(MEASUREMENTS)
comparison["total_tokens"] = comparison["input_tokens"] + comparison["output_tokens"]
comparison

## 3. Filter by release quality before ranking cost

Quality per dollar is a decision aid, never a release gate. Missing cost remains unknown. A candidate with incomplete coverage can be high quality but cannot win a cost comparison.

In [ ]:
comparison["quality_eligible"] = (
    (comparison["quality_score"] >= 0.90)
    & (comparison["minimum_row_quality"] >= 0.90)
    & (comparison["critical_case_pass_rate"] == 1.0)
    & (comparison["recommendation_policy_compliance"] == 1.0)
)
comparison["cost_comparable"] = (
    comparison["quality_eligible"]
    & (comparison["cost_coverage"] == 1.0)
    & comparison["target_inference_cost_usd"].notna()
)
comparison["quality_per_cost"] = comparison["quality_score"].div(
    comparison["target_inference_cost_usd"]
)

eligible = comparison.loc[comparison["cost_comparable"]].copy()
DEMONSTRATION_COST_BUDGET_USD = 0.005
within_budget = eligible.loc[
    eligible["target_inference_cost_usd"] <= DEMONSTRATION_COST_BUDGET_USD
]
preferred_under_budget = (
    None
    if within_budget.empty
    else within_budget.sort_values(
        ["quality_score", "target_inference_cost_usd"],
        ascending=[False, True],
    ).iloc[0]["logical_model"]
)

comparison[
    [
        "logical_model",
        "quality_eligible",
        "cost_comparable",
        "quality_score",
        "target_inference_cost_usd",
        "evaluation_judge_cost_usd",
        "cost_coverage",
        "quality_per_cost",
    ]
]

In [ ]:
unknown_cost_models = comparison.loc[
    comparison["quality_eligible"] & ~comparison["cost_comparable"],
    "logical_model",
].tolist()
{
    "preferred_under_demonstration_budget": preferred_under_budget,
    "unknown_cost_models": unknown_cost_models,
    "decision": "inconclusive",
    "release": "blocked_until_connected_evaluation",
    "reason": (
        "the table is a simulated fixture; rerun the exact contract against "
        "configured logical models and authoritative cost evidence"
    ),
}

## 4. Connected comparison checklist

Before any billable request:

1. Require two configured logical model names and resolve both with `context.providers.model(...)`; fail on missing capabilities first.
2. Load one exact prompt version inside every traced prediction.
3. Use the same ordered cases, scorer versions, judge model, inference parameters, and conservative evaluation concurrency.
4. Read latency, input/output tokens, `trace.info.cost`, and cost coverage from completed traces. Account separately for target predictions and judge calls; MLflow evaluation may make an additional prediction while validating tracing.
5. Treat unavailable cost as `None`, not zero. Prefer gateway or billing records for chargeback.
6. Apply absolute quality, policy, critical-row, latency, token, and cost gates. Record a recommendation; do not provision endpoints or mutate deployment aliases from this notebook.